# DORAnet → enzyme hypotheses → DNA design dossier

This notebook converts DORAnet-generated reaction networks into concise downstream design tables: parsed reactions, enzyme hypotheses, enzyme-candidate templates, and non-operational DNA design plans for expert review.

In [26]:
from pathlib import Path
import glob
import json
import os
import re
import textwrap
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from rdkit import Chem

In [34]:
dataDir = '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/'
resultsDir = os.path.join(dataDir, 'Results/combinedEbolaVirus_bestMACAW_allDB_generative/')
DORANETmoleculesDataDir = os.path.join(resultsDir, 'DORAnet/InoAdeGuoXan/doranet_output_gen3/')
DNADesignResultsDir = os.path.join(resultsDir, 'DNA_design/')
os.makedirs(DNADesignResultsDir, exist_ok=True)

In [35]:
# Fill this template later after enzyme database/literature review.
#enzymeCandidatesCsv = DNADesignResultsDir + "enzymeCandidateTemplate_filled.csv"

hostSystem = "cell_free"  # E_coli_K12_or_cell_free
maxEnzymeCandidatesPerStep = 3

## 1. Discover DORAnet JSON files

In [36]:
DORANETmoleculesDataDir = Path(DORANETmoleculesDataDir)
def starterNumFromName(dirName):
    match = re.search(r"starter_(\d+)", dirName)
    return int(match.group(1)) if match else -1

allStarterDirPaths = sorted(
    [p for p in DORANETmoleculesDataDir.glob("starter_*") if p.is_dir()],
    key=lambda p: starterNumFromName(p.name),
)

fileInfoList = []
missingJsonDirs = []

for starterDirPath in allStarterDirPaths:
    dirName = starterDirPath.name
    expectedJsonPath = starterDirPath / f"{dirName}_network_pretreated.json"

    if expectedJsonPath.exists():
        jsonPath = expectedJsonPath
    else:
        fallbackJsonPaths = sorted(starterDirPath.glob("*_network_pretreated.json"))
        jsonPath = fallbackJsonPaths[0] if fallbackJsonPaths else None

    if jsonPath is None:
        missingJsonDirs.append(dirName)
        continue

    fileInfoList.append({
        "dirName": dirName,
        "dirPath": str(starterDirPath),
        "jsonPath": str(jsonPath),
        "starterNum": starterNumFromName(dirName),
    })

print(f"Starter directories found : {len(allStarterDirPaths)}")
print(f"JSON files found          : {len(fileInfoList)}")
print(f"Missing JSON directories  : {len(missingJsonDirs)}")
fileInfoList[:3]

Starter directories found : 20
JSON files found          : 18
Missing JSON directories  : 2


[{'dirName': 'starter_00000',
  'dirPath': '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DORAnet/InoAdeGuoXan/doranet_output_gen3/starter_00000',
  'jsonPath': '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DORAnet/InoAdeGuoXan/doranet_output_gen3/starter_00000/starter_00000_gen3_network_pretreated.json',
  'starterNum': 0},
 {'dirName': 'starter_00001',
  'dirPath': '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DORAnet/InoAdeGuoXan/doranet_output_gen3/starter_00001',
  'jsonPath': '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DORAnet/InoAdeGuoXan/doranet_output_gen3/starter_00001/starter_00001_gen3_network_pretreated.json',
  'starterNum': 1},
 {'dirName': 'starter_00002',
  'dirPath': '/mnt

## 2. Parse DORAnet reaction strings

In [37]:
def splitMoleculeString(moleculeString):
    return [mol for mol in str(moleculeString).split(".") if mol]

def parseDoranetReaction(rxnString, fileInfo):
    parts = str(rxnString).split(">")
    if len(parts) != 4:
        raise ValueError(f"Expected 4 fields separated by '>'; found {len(parts)}")

    reactants, ruleName, metaBlock, products = parts
    metaParts = (metaBlock.split("$") + [None, None, None, None])[:4]
    thermo, reactantStoich, productStoich, reactionType = metaParts

    return {
        "reactants": reactants,
        "products": products,
        "reactionString": f"{reactants} >> {products}",
        "ruleName": ruleName,
        "thermo": thermo,
        "reactantStoich": reactantStoich,
        "productStoich": productStoich,
        "reactionType": reactionType,
        "numReactantMolecules": len(splitMoleculeString(reactants)),
        "numProductMolecules": len(splitMoleculeString(products)),
        "SourceStarterNum": fileInfo["starterNum"],
        "SourceDirectory": fileInfo["dirName"],
    }

reactionRecords = []
jsonReadErrors = []
uniqueReactantMolecules = set()
uniqueProductMolecules = set()

for fileInfo in tqdm(fileInfoList, desc="Reading DORAnet JSON"):
    try:
        with open(fileInfo["jsonPath"], "r", encoding="utf-8") as f:
            reactionList = json.load(f)

        for rxnString in reactionList:
            record = parseDoranetReaction(rxnString, fileInfo)
            reactionRecords.append(record)
            uniqueReactantMolecules.update(splitMoleculeString(record["reactants"]))
            uniqueProductMolecules.update(splitMoleculeString(record["products"]))

    except Exception as exc:
        jsonReadErrors.append({**fileInfo, "error": str(exc)})

if not reactionRecords:
    raise RuntimeError("No reactions loaded. Check DORANETmoleculesDataDir and JSON format.")

reactionDF = pd.DataFrame(reactionRecords).reset_index(drop=True)
reactionDF.to_csv(DNADesignResultsDir + "reactionDF.csv", index=False)

print(f"Reactions loaded     : {len(reactionDF):,}")
print(f"Failed JSON files    : {len(jsonReadErrors):,}")
print(f"Unique reactant mols : {len(uniqueReactantMolecules):,}")
print(f"Unique product mols  : {len(uniqueProductMolecules):,}")
reactionDF.head()

Reading DORAnet JSON: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 18/18 [00:01<00:00, 13.65it/s]


Reactions loaded     : 116,827
Failed JSON files    : 0
Unique reactant mols : 3,060
Unique product mols  : 44,107


,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,SourceStarterNum,SourceDirectory
0,CCC(=O)OP(=O)(O)OC(C)O.O=P(O)(O)O,CC(O)O.CCC(=O)OP(=O)(O)OP(=O)(O)O,CCC(=O)OP(=O)(O)OC(C)O.O=P(O)(O)O >> CC(O)O.CC...,rule0768_2,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,4,starter_00004
1,CC(=O)OP(=O)(O)OC(O)C(O)C(=O)O.CC(=O)OP(=O)(O)...,CC(=O)OP(=O)(O)OC(O)C(=O)C(=O)O.CC(=O)OP(=O)(O...,CC(=O)OP(=O)(O)OC(O)C(O)C(=O)O.CC(=O)OP(=O)(O)...,rule0324_2,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,4,starter_00004
2,O=P1(O)OC(O)COC(CO)O1.O=P1(O)OC(O)COC(CO)O1,O=P1(O)OC(CO)OCC(OC(O)CO)O1.O=P1(O)OCC(O)O1,O=P1(O)OC(O)COC(CO)O1.O=P1(O)OC(O)COC(CO)O1 >>...,rule0384_1,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,4,starter_00004
3,CC(=O)OP(=O)(O)OC(O)C(C)O.NC(=O)c1ccc[n+]([C@@...,CC(=O)OP(=O)(O)OC(=O)C(C)O.NC(=O)C1=CN([C@@H]2...,CC(=O)OP(=O)(O)OC(O)C(C)O.NC(=O)c1ccc[n+]([C@@...,rule0002_144,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,4,starter_00004
4,CCC(=O)OP(=O)(O)OC(=O)C(=O)O.OO,CCC(=O)OP(=O)(O)OC(=O)C(O)O.O=O,CCC(=O)OP(=O)(O)OC(=O)C(=O)O.OO >> CCC(=O)OP(=...,rule0078_15,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,4,starter_00004


## 3. Build molecule table and optional canonical SMILES

In [38]:
def canonicalizeSmiles(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    return Chem.MolToSmiles(mol, canonical=True) if mol is not None else None

moleculeRecords = []
for smiles in sorted(uniqueReactantMolecules | uniqueProductMolecules):
    moleculeRecords.append({
        "SMILES": smiles,
        "Canonical_SMILES": canonicalizeSmiles(smiles),
        "appearsAsReactant": smiles in uniqueReactantMolecules,
        "appearsAsProduct": smiles in uniqueProductMolecules,
    })

moleculeDF = pd.DataFrame(moleculeRecords)
moleculeDF.to_csv(DNADesignResultsDir + "moleculeDF.csv", index=False)
moleculeDF.head()

,SMILES,Canonical_SMILES,appearsAsReactant,appearsAsProduct
0,*C1=C(*)C(=O)C(*)=C(*)C1=O,*C1=C(*)C(=O)C(*)=C(*)C1=O,True,True
1,*c1c(*)c(O)c(*)c(*)c1O,*c1c(*)c(O)c(*)c(*)c1O,True,True
2,C,C,False,True
3,C#N,C#N,True,True
4,C1CO1,C1CO1,False,True


## 4. Choose route-step input

Use your curated recommended pathway table if available. Otherwise, the parsed reaction network is used as a step-level design database.

In [39]:
# Build route-step table directly from parsed DORAnet reaction network

routeStepDF = reactionDF.copy()

routeStepDF["routeId"] = "network_" + routeStepDF["SourceDirectory"].astype(str)
routeStepDF["stepId"]  = routeStepDF.groupby("routeId").cumcount() + 1

requiredCols = [
    "routeId",
    "stepId",
    "reactants",
    "products",
    "reactionString",
    "ruleName",
    "reactionType",
    "thermo",
    "reactantStoich",
    "productStoich",
    "SourceStarterNum",
    "SourceDirectory",
]

for col in requiredCols:
    if col not in routeStepDF.columns:
        routeStepDF[col] = ""

routeStepDF = routeStepDF[requiredCols].copy()

routeStepDF.to_csv(DNADesignResultsDir + "routeStepDF.csv", index=False)

print(f"Route step rows: {len(routeStepDF):,}")
routeStepDF.head()

Route step rows: 116,827


,routeId,stepId,reactants,products,reactionString,ruleName,reactionType,thermo,reactantStoich,productStoich,SourceStarterNum,SourceDirectory
0,network_starter_00004,1,CCC(=O)OP(=O)(O)OC(C)O.O=P(O)(O)O,CC(O)O.CCC(=O)OP(=O)(O)OP(=O)(O)O,CCC(=O)OP(=O)(O)OC(C)O.O=P(O)(O)O >> CC(O)O.CC...,rule0768_2,Enzymatic,No_Thermo,"(1, 1)","(1, 1)",4,starter_00004
1,network_starter_00004,2,CC(=O)OP(=O)(O)OC(O)C(O)C(=O)O.CC(=O)OP(=O)(O)...,CC(=O)OP(=O)(O)OC(O)C(=O)C(=O)O.CC(=O)OP(=O)(O...,CC(=O)OP(=O)(O)OC(O)C(O)C(=O)O.CC(=O)OP(=O)(O)...,rule0324_2,Enzymatic,No_Thermo,"(1, 1)","(1, 1)",4,starter_00004
2,network_starter_00004,3,O=P1(O)OC(O)COC(CO)O1.O=P1(O)OC(O)COC(CO)O1,O=P1(O)OC(CO)OCC(OC(O)CO)O1.O=P1(O)OCC(O)O1,O=P1(O)OC(O)COC(CO)O1.O=P1(O)OC(O)COC(CO)O1 >>...,rule0384_1,Enzymatic,No_Thermo,"(1, 1)","(1, 1)",4,starter_00004
3,network_starter_00004,4,CC(=O)OP(=O)(O)OC(O)C(C)O.NC(=O)c1ccc[n+]([C@@...,CC(=O)OP(=O)(O)OC(=O)C(C)O.NC(=O)C1=CN([C@@H]2...,CC(=O)OP(=O)(O)OC(O)C(C)O.NC(=O)c1ccc[n+]([C@@...,rule0002_144,Enzymatic,No_Thermo,"(1, 1)","(1, 1)",4,starter_00004
4,network_starter_00004,5,CCC(=O)OP(=O)(O)OC(=O)C(=O)O.OO,CCC(=O)OP(=O)(O)OC(=O)C(O)O.O=O,CCC(=O)OP(=O)(O)OC(=O)C(=O)O.OO >> CCC(=O)OP(=...,rule0078_15,Enzymatic,No_Thermo,"(1, 1)","(1, 1)",4,starter_00004


## 5. Rank routes if potency / toxicity / ADME columns are present

In [40]:
reference_compounds = (
    pd.read_csv(resultsDir + "/Top20_InoAdeGuoXan_DORAnetGenerated_ARTprediction.csv")
    .nlargest(20, "pPotency_prediction")
    .reset_index(drop=True)
)
reference_compounds

,Starter_Compound_SMILES,Canonical_SMILES,DORAnet_gen,source_DataBase,pPotency_prediction,pPotency_std,pPotency_lower_95CI,pPotency_upper_95CI,IC50(M)_prediction,IC50(M)_lower_95CI,IC50(M)_upper_95CI
0,O=C(O)CC(=O)OP(=O)(O)O,O=C(OP(=O)(O)O)C(=O)OP(=O)(O)O,2,Top20_InoAdeGuoXan,5.830348,0.476798,4.895824,6.764872,0.000001,1.718416e-07,0.000013
1,O=C(O)CC(=O)OP(=O)(O)O,O=C1OC1(OC(O)O)C(=O)OP(=O)(O)O,3,Top20_InoAdeGuoXan,5.821484,0.477988,4.884627,6.758341,0.000002,1.744451e-07,0.000013
2,O=C(O)CC(=O)OP(=O)(O)O,O=CC(=O)C(O)(OP(=O)(O)O)C(=O)C=O,3,Top20_InoAdeGuoXan,5.820582,0.477015,4.885634,6.755531,0.000002,1.755777e-07,0.000013
3,O=C(O)CC(=O)OP(=O)(O)O,O=CC(C=O)(C=O)C(=O)OP(=O)(O)O,3,Top20_InoAdeGuoXan,5.817794,0.478980,4.878993,6.756595,0.000002,1.751480e-07,0.000013
4,O=C(O)CC(=O)OP(=O)(O)O,O=C1OC1(CC(O)(O)O)C(=O)OP(=O)(O)O,3,Top20_InoAdeGuoXan,5.817628,0.477515,4.881699,6.753557,0.000002,1.763775e-07,0.000013
5,O=C(O)CC(=O)OP(=O)(O)O,O=CC(=O)OP(=O)(O)OC(=O)C=O,2,Top20_InoAdeGuoXan,5.816470,0.477022,4.881507,6.751433,0.000002,1.772423e-07,0.000013
6,CC(=O)OP(=O)(O)OP(=O)(O)O,O=C(O)COP(=O)(OP(=O)(O)O)OP(=O)(O)O,3,Top20_InoAdeGuoXan,5.812589,0.476426,4.878795,6.746383,0.000002,1.793150e-07,0.000013
7,CC(=O)OP(=O)(O)OP(=O)(O)O,O=CC(=O)OP(=O)(OP(=O)(O)O)OP(=O)(O)O,3,Top20_InoAdeGuoXan,5.810244,0.476181,4.876931,6.743558,0.000002,1.804853e-07,0.000013
8,CC(=O)CC(=O)O,CC(O)(C=O)C(=O)CC(=O)O,3,Top20_InoAdeGuoXan,5.806817,0.475066,4.875688,6.737946,0.000002,1.828326e-07,0.000013
9,O=C(O)CC(=O)OP(=O)(O)O,O=C(O)C1(C(=O)OP(=O)(O)O)CC(O)(O)O1,3,Top20_InoAdeGuoXan,5.801140,0.476524,4.867153,6.735126,0.000002,1.840237e-07,0.000014


In [41]:
def minMax01(series, higherIsBetter=True):
    values = pd.to_numeric(series, errors="coerce")
    if values.notna().sum() == 0 or values.max() == values.min():
        return pd.Series(np.nan, index=series.index)

    scaled = (values - values.min()) / (values.max() - values.min())
    return scaled if higherIsBetter else 1 - scaled


# --------------------------------------------------
# Filter route steps using reference_compounds
# --------------------------------------------------

referenceSmilesSet = set(
    reference_compounds["Canonical_SMILES"]
    .dropna()
    .astype(str)
    .str.strip()
)

routeStepFilteredDF = routeStepDF.copy()

if "Canonical_SMILES" in routeStepFilteredDF.columns:
    routeStepFilteredDF["Canonical_SMILES"] = (
        routeStepFilteredDF["Canonical_SMILES"]
        .astype(str)
        .str.strip()
    )

    routeStepFilteredDF = routeStepFilteredDF[
        routeStepFilteredDF["Canonical_SMILES"].isin(referenceSmilesSet)
    ].copy()

else:
    # Fallback: use individual product molecules from DORAnet products column
    routeStepFilteredDF["productMoleculeList"] = routeStepFilteredDF["products"].apply(splitMoleculeString)

    routeStepFilteredDF = routeStepFilteredDF.explode("productMoleculeList").reset_index(drop=True)
    routeStepFilteredDF["Canonical_SMILES"] = (
        routeStepFilteredDF["productMoleculeList"]
        .astype(str)
        .str.strip()
    )

    routeStepFilteredDF = routeStepFilteredDF[
        routeStepFilteredDF["Canonical_SMILES"].isin(referenceSmilesSet)
    ].copy()

    routeStepFilteredDF = routeStepFilteredDF.drop(columns=["productMoleculeList"])


print(f"Original route-step rows : {len(routeStepDF):,}")
print(f"Filtered route-step rows : {len(routeStepFilteredDF):,}")
print(f"Reference compounds used : {len(referenceSmilesSet):,}")


# --------------------------------------------------
# Score only filtered route steps
# --------------------------------------------------

routeScoreDF = routeStepFilteredDF.groupby("routeId").agg(
    routeLength=("stepId", "count"),
    finalProduct=("products", "last"),
    finalCanonicalSMILES=("Canonical_SMILES", "last"),
).reset_index()


if "pPotency_prediction" in routeStepFilteredDF.columns:
    potencyByRoute = routeStepFilteredDF.groupby("routeId")["pPotency_prediction"].max()
    routeScoreDF = routeScoreDF.merge(
        potencyByRoute.rename("bestPotency").reset_index(),
        on="routeId",
        how="left"
    )
    routeScoreDF["potencyScore01"] = minMax01(routeScoreDF["bestPotency"], True)


if "coreToxicityScore" in routeStepFilteredDF.columns:
    toxByRoute = routeStepFilteredDF.groupby("routeId")["coreToxicityScore"].min()
    routeScoreDF = routeScoreDF.merge(
        toxByRoute.rename("bestToxicity").reset_index(),
        on="routeId",
        how="left"
    )
    routeScoreDF["toxicityScore01"] = minMax01(routeScoreDF["bestToxicity"], False)


if "ADMEFeasibility" in routeStepFilteredDF.columns:
    admeByRoute = routeStepFilteredDF.groupby("routeId")["ADMEFeasibility"].max()
    routeScoreDF = routeScoreDF.merge(
        admeByRoute.rename("bestADME").reset_index(),
        on="routeId",
        how="left"
    )
    routeScoreDF["admeScore01"] = minMax01(routeScoreDF["bestADME"], True)


scoreCols = [
    col for col in ["potencyScore01", "toxicityScore01", "admeScore01"]
    if col in routeScoreDF.columns
]

if scoreCols:
    routeScoreDF["routePriorityScore"] = routeScoreDF[scoreCols].mean(axis=1)
else:
    routeScoreDF["routePriorityScore"] = 1 / routeScoreDF["routeLength"].clip(lower=1)


routeScoreDF = (
    routeScoreDF
    .sort_values("routePriorityScore", ascending=False)
    .reset_index(drop=True)
)


# Save outputs
routeStepFilteredDF.to_csv(DNADesignResultsDir + "routeStepFilteredDF.csv", index=False)
routeScoreDF.to_csv(DNADesignResultsDir + "routeScoreDF.csv", index=False)

routeScoreDF.head(10)

Original route-step rows : 116,827
Filtered route-step rows : 44
Reference compounds used : 20


,routeId,routeLength,finalProduct,finalCanonicalSMILES,routePriorityScore
0,network_starter_00004,2,CC(=O)OP(=O)(O)OC(=O)C1OC(O)O1,CC(=O)OP(=O)(O)OC(=O)C1OC(O)O1,0.500000
1,network_starter_00018,2,CC(O)(C=O)C(=O)CC(=O)O,CC(O)(C=O)C(=O)CC(=O)O,0.500000
2,network_starter_00006,11,O=CC(=O)OP(=O)(O)O.O=CC(=O)OP(=O)(OP(=O)(O)O)O...,O=CC(=O)OP(=O)(OP(=O)(O)O)OP(=O)(O)O,0.090909
3,network_starter_00009,29,O=CC(C=O)(C=O)C(=O)OP(=O)(O)O.O=CCC(=O)OP(=O)(O)O,O=CC(C=O)(C=O)C(=O)OP(=O)(O)O,0.034483


## 6. Map reaction steps to enzyme hypotheses

In [42]:
enzymeRuleMap = [
    (["methyl", "methylation"], "methyltransferase", "2.1.1.-", "SAM"),
    (["glycosyl", "glucosyl", "sugar"], "glycosyltransferase", "2.4.-.-", "UDP-sugar"),
    (["phosph", "kinase"], "kinase / phosphotransferase", "2.7.-.-", "ATP"),
    (["oxid", "hydroxyl", "monooxygenase"], "oxidoreductase / monooxygenase", "1.14.-.-", "NAD(P)H, O2"),
    (["reduct", "dehydrogen"], "reductase / dehydrogenase", "1.-.-.-", "NAD(P)H"),
    (["hydrolysis", "hydrolase"], "hydrolase", "3.-.-.-", "H2O"),
    (["amide", "acyl", "ligase"], "ligase / acyltransferase", "6.-.-.- or 2.3.-.-", "ATP or acyl-CoA"),
    (["transamin", "amination", "amine"], "aminotransferase", "2.6.-.-", "PLP"),
    (["carbox", "decarbox"], "carboxylase / decarboxylase", "4.1.-.- or 6.4.-.-", "CO2 / biotin / ATP"),
]

def inferEnzymeHypothesis(ruleName, reactionType):
    text = f"{ruleName} {reactionType}".lower()
    for keywords, enzymeClass, ecHint, cofactorHint in enzymeRuleMap:
        if any(keyword in text for keyword in keywords):
            return enzymeClass, ecHint, cofactorHint, "keyword_match"
    return "manual_review_required", "unknown", "unknown", "no_keyword_match"

def buildEnzymeSearchQuery(row):
    return " | ".join([
        f"enzyme class: {row['enzymeClass']}",
        f"EC hint: {row['ecHint']}",
        f"reaction type: {row['reactionType']}",
        f"rule: {row['ruleName']}",
        f"reactants: {str(row['reactants'])[:120]}",
        f"products: {str(row['products'])[:120]}",
    ])

enzymeHypothesisDF = routeStepDF.copy()
hypothesisRows = enzymeHypothesisDF.apply(
    lambda row: inferEnzymeHypothesis(row.get("ruleName", ""), row.get("reactionType", "")), axis=1
)
enzymeHypothesisDF[["enzymeClass", "ecHint", "cofactorHint", "inferenceMethod"]] = pd.DataFrame(
    hypothesisRows.tolist(), index=enzymeHypothesisDF.index
)
enzymeHypothesisDF["enzymeSearchQuery"] = enzymeHypothesisDF.apply(buildEnzymeSearchQuery, axis=1)
enzymeHypothesisDF["manualReviewRequired"] = enzymeHypothesisDF["enzymeClass"].eq("manual_review_required")

enzymeHypothesisDF.to_csv(DNADesignResultsDir + "enzymeHypothesisDF.csv", index=False)
enzymeHypothesisDF[["routeId", "stepId", "reactionType", "ruleName", "enzymeClass", "ecHint", "cofactorHint", "manualReviewRequired"]].head(10)

,routeId,stepId,reactionType,ruleName,enzymeClass,ecHint,cofactorHint,manualReviewRequired
0,network_starter_00004,1,Enzymatic,rule0768_2,manual_review_required,unknown,unknown,True
1,network_starter_00004,2,Enzymatic,rule0324_2,manual_review_required,unknown,unknown,True
2,network_starter_00004,3,Enzymatic,rule0384_1,manual_review_required,unknown,unknown,True
3,network_starter_00004,4,Enzymatic,rule0002_144,manual_review_required,unknown,unknown,True
4,network_starter_00004,5,Enzymatic,rule0078_15,manual_review_required,unknown,unknown,True
5,network_starter_00004,6,Enzymatic,rule0549_1,manual_review_required,unknown,unknown,True
6,network_starter_00004,7,Enzymatic,rule0007_176,manual_review_required,unknown,unknown,True
7,network_starter_00004,8,Enzymatic,rule0169_1,manual_review_required,unknown,unknown,True
8,network_starter_00004,9,Enzymatic,rule0169_1,manual_review_required,unknown,unknown,True
9,network_starter_00004,10,Enzymatic,rule0164_2,manual_review_required,unknown,unknown,True


## 7. Create enzyme candidate template

Fill this CSV after searching UniProt, BRENDA, Rhea, KEGG, MetaCyc, and literature. Keep one row per candidate enzyme per pathway step.

In [44]:
candidateTemplateCols = [
    "routeId", "stepId", "reactants", "products", "reactionType", "ruleName",
    "enzymeClass", "ecHint", "cofactorHint", "enzymeSearchQuery",
]

enzymeCandidateTemplateDF = enzymeHypothesisDF[candidateTemplateCols].copy()

enzymeCandidateTemplateDF["enzymeName"] = "TO_FILL"
enzymeCandidateTemplateDF["ecNumber"] = "TO_FILL"
enzymeCandidateTemplateDF["sourceOrganism"] = "TO_FILL"
enzymeCandidateTemplateDF["proteinAccession"] = "TO_FILL"
enzymeCandidateTemplateDF["proteinSequenceAvailable"] = False
enzymeCandidateTemplateDF["evidenceNotes"] = "TO_FILL"
enzymeCandidateTemplateDF["evidenceScore"] = np.nan
enzymeCandidateTemplateDF["substrateScopeScore"] = np.nan
enzymeCandidateTemplateDF["hostCompatibilityScore"] = np.nan
enzymeCandidateTemplateDF["expressionPrecedentScore"] = np.nan
enzymeCandidateTemplateDF["biosecurityFlag"] = "review_required"
enzymeCandidateTemplateDF["selectedForDesign"] = False

templatePath = DNADesignResultsDir + "enzymeCandidateTemplate.csv"
enzymeCandidateTemplateDF.to_csv(templatePath, index=False)
print(templatePath)
enzymeCandidateTemplateDF.head()

/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DNA_design/enzymeCandidateTemplate.csv


,routeId,stepId,reactants,products,reactionType,ruleName,enzymeClass,ecHint,cofactorHint,enzymeSearchQuery,...,sourceOrganism,proteinAccession,proteinSequenceAvailable,evidenceNotes,evidenceScore,substrateScopeScore,hostCompatibilityScore,expressionPrecedentScore,biosecurityFlag,selectedForDesign
0,network_starter_00004,1,CCC(=O)OP(=O)(O)OC(C)O.O=P(O)(O)O,CC(O)O.CCC(=O)OP(=O)(O)OP(=O)(O)O,Enzymatic,rule0768_2,manual_review_required,unknown,unknown,enzyme class: manual_review_required | EC hint...,...,TO_FILL,TO_FILL,False,TO_FILL,NaN,NaN,NaN,NaN,review_required,False
1,network_starter_00004,2,CC(=O)OP(=O)(O)OC(O)C(O)C(=O)O.CC(=O)OP(=O)(O)...,CC(=O)OP(=O)(O)OC(O)C(=O)C(=O)O.CC(=O)OP(=O)(O...,Enzymatic,rule0324_2,manual_review_required,unknown,unknown,enzyme class: manual_review_required | EC hint...,...,TO_FILL,TO_FILL,False,TO_FILL,NaN,NaN,NaN,NaN,review_required,False
2,network_starter_00004,3,O=P1(O)OC(O)COC(CO)O1.O=P1(O)OC(O)COC(CO)O1,O=P1(O)OC(CO)OCC(OC(O)CO)O1.O=P1(O)OCC(O)O1,Enzymatic,rule0384_1,manual_review_required,unknown,unknown,enzyme class: manual_review_required | EC hint...,...,TO_FILL,TO_FILL,False,TO_FILL,NaN,NaN,NaN,NaN,review_required,False
3,network_starter_00004,4,CC(=O)OP(=O)(O)OC(O)C(C)O.NC(=O)c1ccc[n+]([C@@...,CC(=O)OP(=O)(O)OC(=O)C(C)O.NC(=O)C1=CN([C@@H]2...,Enzymatic,rule0002_144,manual_review_required,unknown,unknown,enzyme class: manual_review_required | EC hint...,...,TO_FILL,TO_FILL,False,TO_FILL,NaN,NaN,NaN,NaN,review_required,False
4,network_starter_00004,5,CCC(=O)OP(=O)(O)OC(=O)C(=O)O.OO,CCC(=O)OP(=O)(O)OC(=O)C(O)O.O=O,Enzymatic,rule0078_15,manual_review_required,unknown,unknown,enzyme class: manual_review_required | EC hint...,...,TO_FILL,TO_FILL,False,TO_FILL,NaN,NaN,NaN,NaN,review_required,False


## 8. Rank filled enzyme candidates

In [45]:
scoreWeights = {
    "evidenceScore": 0.35,
    "substrateScopeScore": 0.30,
    "hostCompatibilityScore": 0.20,
    "expressionPrecedentScore": 0.15,
}

if Path(enzymeCandidatesCsv).exists():
    enzymeCandidateDF = pd.read_csv(enzymeCandidatesCsv)
    for col in scoreWeights:
        enzymeCandidateDF[col] = pd.to_numeric(enzymeCandidateDF.get(col, 0), errors="coerce").fillna(0)

    enzymeCandidateDF["enzymeConfidenceScore"] = sum(
        scoreWeights[col] * enzymeCandidateDF[col] for col in scoreWeights
    )
    enzymeCandidateDF["blockedByBiosecurityFlag"] = enzymeCandidateDF["biosecurityFlag"].astype(str).str.lower().ne("clear")

    selectedEnzymeDF = (
        enzymeCandidateDF
        .query("blockedByBiosecurityFlag == False")
        .sort_values(["routeId", "stepId", "enzymeConfidenceScore"], ascending=[True, True, False])
        .groupby(["routeId", "stepId"], as_index=False)
        .head(maxEnzymeCandidatesPerStep)
        .reset_index(drop=True)
    )
else:
    enzymeCandidateDF = enzymeCandidateTemplateDF.copy()
    enzymeCandidateDF["enzymeConfidenceScore"] = np.nan
    selectedEnzymeDF = enzymeCandidateDF.copy()

if not selectedEnzymeDF.empty:
    selectedEnzymeDF["candidateRank"] = selectedEnzymeDF.groupby(["routeId", "stepId"]).cumcount() + 1

selectedEnzymeDF.to_csv(DNADesignResultsDir + "selectedEnzymeDF.csv", index=False)
selectedEnzymeDF.head()

NameError: name 'enzymeCandidatesCsv' is not defined

## 9. Generate non-operational DNA design plan

This table records the expression-construct strategy for review. It intentionally does not generate synthesis-ready DNA sequences.

In [ ]:
def promoterPlanForHost(hostSystem):
    if "cell_free" in hostSystem.lower():
        return "cell-free-compatible expression control; tune after single-enzyme test"
    if "coli" in hostSystem.lower():
        return "medium inducible bacterial promoter; tune by promoter/RBS library"
    if "yeast" in hostSystem.lower():
        return "yeast-compatible promoter; tune by promoter copy/strength"
    return "host-appropriate promoter; tune experimentally"

def vectorPlanForStep(stepCount):
    if stepCount <= 1:
        return "single-enzyme validation construct"
    if stepCount <= 3:
        return "modular multi-enzyme construct after single-step validation"
    return "split pathway into modules before full assembly"

routeLengthMap = routeStepDF.groupby("routeId")["stepId"].count().to_dict()

dnaDesignRecords = []
for _, row in selectedEnzymeDF.iterrows():
    routeId = row["routeId"]
    stepId = row["stepId"]
    candidateRank = int(row.get("candidateRank", 1)) if pd.notna(row.get("candidateRank", 1)) else 1
    routeLength = routeLengthMap.get(routeId, np.nan)

    dnaDesignRecords.append({
        "constructId": f"{routeId}_step{stepId}_cand{candidateRank}",
        "routeId": routeId,
        "stepId": stepId,
        "candidateRank": candidateRank,
        "hostSystem": hostSystem,
        "enzymeName": row.get("enzymeName", "TO_FILL"),
        "proteinAccession": row.get("proteinAccession", "TO_FILL"),
        "enzymeClass": row.get("enzymeClass", ""),
        "cofactorHint": row.get("cofactorHint", ""),
        "constructPurpose": "enzyme expression feasibility review",
        "promoterPlan": promoterPlanForHost(hostSystem),
        "fivePrimeControlPlan": "host-appropriate RBS/5'UTR; tune after enzyme choice",
        "codingSequencePolicy": "retrieve native CDS or codon-optimize only after biosafety and IP review",
        "terminatorPlan": "standard host-compatible terminator",
        "vectorPlan": vectorPlanForStep(routeLength),
        "sequenceGenerated": False,
        "synthesisReady": False,
        "reviewStatus": "biosafety_and_domain_expert_review_required",
        "notes": "Non-operational design record; no raw synthesis-ready DNA sequence generated.",
    })

dnaDesignPlanDF = pd.DataFrame(dnaDesignRecords)
dnaDesignPlanDF.to_csv(DNADesignResultsDir + "dnaDesignPlanDF.csv", index=False)
dnaDesignPlanDF.head()

## 10. Write route design cards

In [ ]:
def compactText(value, width=90):
    return textwrap.shorten(str(value), width=width, placeholder="...")

cardLines = ["# DORAnet downstream DNA design cards", ""]

for routeId, stepGroupDF in enzymeHypothesisDF.groupby("routeId"):
    scoreRowDF = routeScoreDF.query("routeId == @routeId")
    scoreText = "not scored"
    if not scoreRowDF.empty:
        scoreText = f"priority={scoreRowDF.iloc[0]['routePriorityScore']:.3f}; length={scoreRowDF.iloc[0]['routeLength']}"

    cardLines += [f"## {routeId}", f"- Route score: {scoreText}", ""]
    for _, stepRow in stepGroupDF.sort_values("stepId").iterrows():
        cardLines += [
            f"### Step {stepRow['stepId']}",
            f"- Reaction: `{compactText(stepRow['reactionString'])}`",
            f"- DORAnet rule: `{compactText(stepRow['ruleName'])}`",
            f"- Enzyme hypothesis: **{stepRow['enzymeClass']}**; EC hint `{stepRow['ecHint']}`; cofactor `{stepRow['cofactorHint']}`",
            f"- Search query: {compactText(stepRow['enzymeSearchQuery'], 160)}",
            "",
        ]

    designRowsDF = dnaDesignPlanDF.query("routeId == @routeId") if not dnaDesignPlanDF.empty else pd.DataFrame()
    if not designRowsDF.empty:
        cardLines += ["### DNA design records", ""]
        for _, designRow in designRowsDF.iterrows():
            cardLines += [
                f"- `{designRow['constructId']}`: {designRow['constructPurpose']}; synthesisReady={designRow['synthesisReady']}; review={designRow['reviewStatus']}",
            ]
        cardLines.append("")

cardsPath = DNADesignResultsDir + "routeDesignCards.md"
cardsPath.write_text("
".join(cardLines), encoding="utf-8")
print(cardsPath)

## 11. Outputs

Core files generated in `DNADesignResultsDir`:

- `reactionDF.csv`: parsed DORAnet reactions
- `moleculeDF.csv`: unique reactant/product molecules
- `routeStepDF.csv`: pathway-step table
- `routeScoreDF.csv`: route-level prioritization
- `enzymeHypothesisDF.csv`: reaction-to-enzyme hypotheses
- `enzymeCandidateTemplate.csv`: template for enzyme database/literature review
- `selectedEnzymeDF.csv`: ranked enzyme candidates if a filled template is provided
- `dnaDesignPlanDF.csv`: non-operational expression-design records
- `routeDesignCards.md`: concise route cards for review